# Tahap 3 — Case Retrieval

Membangun vektor representasi dokumen dan model retrieval menggunakan dua pendekatan:
1. **TF-IDF** + SVM, Naive Bayes, Cosine Similarity
2. **IndoBERT** Embedding + Cosine Similarity (berjalan di CPU)

**Input**: `data/processed/cases.json`

**Output**:
- `data/eval/queries.json` — query uji beserta top-5 hasil retrieval
- `data/eval/retrieval_metrics.csv` — metrik evaluasi retrieval
- `models/tfidf_vectorizer.pkl`, `models/svm_pasal.pkl`, `models/nb_pasal.pkl`
- `models/indobert_embeddings.npy`

## 3.1 Import Library

In [1]:
import json
import pickle
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")

## 3.2 Konfigurasi dan Load Data

In [2]:
DATA_PATH = Path("../data/processed/cases.json")
EVAL_DIR  = Path("../data/eval")
MODEL_DIR = Path("../models")

for d in [EVAL_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cpu")
print(f"Device yang digunakan: {device} (dipaksa CPU agar stabil)")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    cases = json.load(f)

print(f"Total kasus: {len(cases)}")
print(f"Distribusi label_pasal: {dict(Counter([c['label_pasal'] for c in cases]))}")

Device yang digunakan: cpu (dipaksa CPU agar stabil)
Total kasus: 60
Distribusi label_pasal: {'pasal_362': 30, 'pasal_363': 30}


## 3.3 Preprocessing dan Splitting Data

In [3]:
def preprocess_text(text):
    text = text.lower().strip()
    text = re.sub(r"disclaimer.*?ext\.318\)", "", text, flags=re.DOTALL)
    text = re.sub(r"halaman \d+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


for c in cases:
    c["text_processed"] = preprocess_text(c["text_full"])

labels_pasal = [c["label_pasal"] for c in cases]
train_cases, test_cases = train_test_split(
    cases, test_size=0.2, stratify=labels_pasal, random_state=42
)

train_texts       = [c["text_processed"] for c in train_cases]
test_texts        = [c["text_processed"] for c in test_cases]
train_labels_pasal = [c["label_pasal"] for c in train_cases]
test_labels_pasal  = [c["label_pasal"] for c in test_cases]

print(f"Train: {len(train_cases)} | Test: {len(test_cases)}")
print(f"Train distribusi: {dict(Counter(train_labels_pasal))}")
print(f"Test distribusi : {dict(Counter(test_labels_pasal))}")

Train: 48 | Test: 12
Train distribusi: {'pasal_363': 24, 'pasal_362': 24}
Test distribusi : {'pasal_362': 6, 'pasal_363': 6}


## 3.4 Fungsi Evaluasi

In [4]:
def evaluate_model(y_true, y_pred, model_name, label_name):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    print(f"\nModel: {model_name} | Label: {label_name}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return {
        "model": model_name, "label": label_name,
        "accuracy": round(acc, 4), "precision": round(prec, 4),
        "recall": round(rec, 4), "f1_score": round(f1, 4),
    }


def retrieval_predict(query_vectors, train_vectors, train_labels, k=5):
    predictions = []
    for i in range(query_vectors.shape[0]):
        q = query_vectors[i:i+1]
        sims = cosine_similarity(q, train_vectors).flatten()
        top_k_idx = sims.argsort()[::-1][:k]
        top_k_labels = [train_labels[j] for j in top_k_idx]
        predictions.append(Counter(top_k_labels).most_common(1)[0][0])
    return predictions


all_metrics = []

## 3.5 Pendekatan 1: TF-IDF + Machine Learning

In [5]:
vectorizer = TfidfVectorizer(
    max_features=5000, ngram_range=(1, 2),
    sublinear_tf=True, min_df=2, max_df=0.95,
)
X_train = vectorizer.fit_transform(train_texts)
X_test  = vectorizer.transform(test_texts)
all_tfidf_vectors = vectorizer.transform([c["text_processed"] for c in cases])

print(f"TF-IDF shape: train={X_train.shape}, test={X_test.shape}")

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print("TF-IDF vectorizer disimpan")

TF-IDF shape: train=(48, 1469), test=(12, 1469)
TF-IDF vectorizer disimpan


In [6]:
svm_pasal = LinearSVC(max_iter=10000, random_state=42, C=1.0)
svm_pasal.fit(X_train, train_labels_pasal)
svm_pred = svm_pasal.predict(X_test)
m = evaluate_model(test_labels_pasal, svm_pred, "TF-IDF + SVM", "label_pasal")
all_metrics.append(m)

with open(MODEL_DIR / "svm_pasal.pkl", "wb") as f:
    pickle.dump(svm_pasal, f)


Model: TF-IDF + SVM | Label: label_pasal
  Accuracy : 0.9167
  Precision: 0.9286
  Recall   : 0.9167
  F1-Score : 0.9161
              precision    recall  f1-score   support

   pasal_362       0.86      1.00      0.92         6
   pasal_363       1.00      0.83      0.91         6

    accuracy                           0.92        12
   macro avg       0.93      0.92      0.92        12
weighted avg       0.93      0.92      0.92        12



In [7]:
nb_pasal = MultinomialNB(alpha=1.0)
nb_pasal.fit(X_train, train_labels_pasal)
nb_pred = nb_pasal.predict(X_test)
m = evaluate_model(test_labels_pasal, nb_pred, "TF-IDF + NaiveBayes", "label_pasal")
all_metrics.append(m)

with open(MODEL_DIR / "nb_pasal.pkl", "wb") as f:
    pickle.dump(nb_pasal, f)


Model: TF-IDF + NaiveBayes | Label: label_pasal
  Accuracy : 0.8333
  Precision: 0.8333
  Recall   : 0.8333
  F1-Score : 0.8333
              precision    recall  f1-score   support

   pasal_362       0.83      0.83      0.83         6
   pasal_363       0.83      0.83      0.83         6

    accuracy                           0.83        12
   macro avg       0.83      0.83      0.83        12
weighted avg       0.83      0.83      0.83        12



In [8]:
tfidf_ret_pred = retrieval_predict(X_test, X_train, train_labels_pasal, k=5)
m = evaluate_model(test_labels_pasal, tfidf_ret_pred, "TF-IDF + Cosine (k=5)", "label_pasal")
all_metrics.append(m)


Model: TF-IDF + Cosine (k=5) | Label: label_pasal
  Accuracy : 0.8333
  Precision: 0.8333
  Recall   : 0.8333
  F1-Score : 0.8333
              precision    recall  f1-score   support

   pasal_362       0.83      0.83      0.83         6
   pasal_363       0.83      0.83      0.83         6

    accuracy                           0.83        12
   macro avg       0.83      0.83      0.83        12
weighted avg       0.83      0.83      0.83        12



## 3.6 Pendekatan 2: IndoBERT Embedding (CPU)

> Model berjalan di CPU untuk menghindari konflik VRAM dengan proses lain. Proses ini memerlukan waktu sekitar 3-10 menit.

In [9]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask_expanded, 1)
    clamped = torch.clamp(mask_expanded.sum(1), min=1e-9)
    return (summed / clamped).detach().numpy()


def get_indobert_embeddings(texts, tokenizer, model, batch_size=8):
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors="pt"
        )
        with torch.no_grad():
            outputs = model(**encoded)
        emb = mean_pooling(outputs, encoded["attention_mask"])
        all_emb.append(emb)
        print(f"  Batch {i//batch_size + 1}/{(len(texts)+batch_size-1)//batch_size} selesai ({i+len(batch)}/{len(texts)} teks)")
    return np.vstack(all_emb)


MODEL_NAME = "indobenchmark/indobert-base-p1"
print(f"Loading {MODEL_NAME} di CPU...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()
print("Model berhasil di-load")

Loading indobenchmark/indobert-base-p1 di CPU...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model berhasil di-load


In [10]:
all_texts = [c["text_processed"] for c in cases]

emb_path = MODEL_DIR / "indobert_embeddings.npy"
if emb_path.exists():
    print("File embedding sudah ada, load dari file...")
    all_emb = np.load(emb_path)
else:
    print(f"Generating embeddings untuk {len(all_texts)} dokumen...")
    all_emb = get_indobert_embeddings(all_texts, tokenizer, bert_model, batch_size=4)
    np.save(emb_path, all_emb)

print(f"Shape embeddings: {all_emb.shape}")

case_ids = [c["case_id"] for c in cases]
train_idx = [case_ids.index(c["case_id"]) for c in train_cases]
test_idx  = [case_ids.index(c["case_id"]) for c in test_cases]

train_emb = all_emb[train_idx]
test_emb  = all_emb[test_idx]

File embedding sudah ada, load dari file...
Shape embeddings: (60, 768)


In [11]:
bert_ret_pred = retrieval_predict(test_emb, train_emb, train_labels_pasal, k=5)
m = evaluate_model(test_labels_pasal, bert_ret_pred, "IndoBERT + Cosine (k=5)", "label_pasal")
all_metrics.append(m)


Model: IndoBERT + Cosine (k=5) | Label: label_pasal
  Accuracy : 0.9167
  Precision: 0.9286
  Recall   : 0.9167
  F1-Score : 0.9161
              precision    recall  f1-score   support

   pasal_362       0.86      1.00      0.92         6
   pasal_363       1.00      0.83      0.91         6

    accuracy                           0.92        12
   macro avg       0.93      0.92      0.92        12
weighted avg       0.93      0.92      0.92        12



## 3.7 Generate queries.json untuk Evaluasi

In [12]:
tfidf_case_ids  = [c["case_id"] for c in cases]


def retrieve_tfidf(query_text, k=5):
    q_vec = vectorizer.transform([preprocess_text(query_text)])
    sims  = cosine_similarity(q_vec, all_tfidf_vectors).flatten()
    top_k = sims.argsort()[::-1][:k]
    return [{"case_id": tfidf_case_ids[i], "label": cases[i]["label_pasal"],
             "similarity": round(float(sims[i]), 4)} for i in top_k]


def retrieve_bert(query_text, k=5):
    q_emb = get_indobert_embeddings([preprocess_text(query_text)], tokenizer, bert_model, batch_size=1)
    sims  = cosine_similarity(q_emb, all_emb).flatten()
    top_k = sims.argsort()[::-1][:k]
    return [{"case_id": tfidf_case_ids[i], "label": cases[i]["label_pasal"],
             "similarity": round(float(sims[i]), 4)} for i in top_k]


queries = []
for i, tc in enumerate(test_cases):
    tfidf_results = retrieve_tfidf(tc["text_processed"], k=5)
    bert_results  = retrieve_bert(tc["text_processed"], k=5)
    queries.append({
        "query_id":               f"q{i+1:02d}",
        "source_case_id":         tc["case_id"],
        "ground_truth_label_pasal": tc["label_pasal"],
        "ground_truth_vonis_bulan": tc["vonis_bulan"],
        "query_text":             tc["text_processed"][:500],
        "tfidf_top5":             tfidf_results,
        "indobert_top5":          bert_results,
    })
    print(f"  q{i+1:02d}: {tc['case_id']} ({tc['label_pasal']}, {tc['vonis_bulan']} bln) "
          f"| TF-IDF top1={tfidf_results[0]['case_id']} | BERT top1={bert_results[0]['case_id']}")

with open(EVAL_DIR / "queries.json", "w", encoding="utf-8") as f:
    json.dump(queries, f, ensure_ascii=False, indent=2)
print(f"\nqueries.json disimpan: {len(queries)} queries")

  Batch 1/1 selesai (1/1 teks)
  q01: case_pasal_362_011 (pasal_362, 12 bln) | TF-IDF top1=case_pasal_362_011 | BERT top1=case_pasal_362_011
  Batch 1/1 selesai (1/1 teks)
  q02: case_pasal_363_029 (pasal_363, 10 bln) | TF-IDF top1=case_pasal_363_029 | BERT top1=case_pasal_363_020
  Batch 1/1 selesai (1/1 teks)
  q03: case_pasal_363_015 (pasal_363, 0 bln) | TF-IDF top1=case_pasal_363_028 | BERT top1=case_pasal_362_004
  Batch 1/1 selesai (1/1 teks)
  q04: case_pasal_362_007 (pasal_362, 5 bln) | TF-IDF top1=case_pasal_362_007 | BERT top1=case_pasal_362_007
  Batch 1/1 selesai (1/1 teks)
  q05: case_pasal_363_016 (pasal_363, 0 bln) | TF-IDF top1=case_pasal_363_016 | BERT top1=case_pasal_362_006
  Batch 1/1 selesai (1/1 teks)
  q06: case_pasal_362_015 (pasal_362, 24 bln) | TF-IDF top1=case_pasal_362_015 | BERT top1=case_pasal_362_015
  Batch 1/1 selesai (1/1 teks)
  q07: case_pasal_363_010 (pasal_363, 36 bln) | TF-IDF top1=case_pasal_363_010 | BERT top1=case_pasal_363_010
  Batch 1/1 sele

## 3.8 Simpan Retrieval Metrics

In [13]:
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(EVAL_DIR / "retrieval_metrics.csv", index=False)
print("\nRETRIEVAL METRICS SUMMARY")
metrics_df


RETRIEVAL METRICS SUMMARY


,model,label,accuracy,precision,recall,f1_score
0,TF-IDF + SVM,label_pasal,0.9167,0.9286,0.9167,0.9161
1,TF-IDF + NaiveBayes,label_pasal,0.8333,0.8333,0.8333,0.8333
2,TF-IDF + Cosine (k=5),label_pasal,0.8333,0.8333,0.8333,0.8333
3,IndoBERT + Cosine (k=5),label_pasal,0.9167,0.9286,0.9167,0.9161
